In [54]:
import pandas as pd
import geopandas as gpd
import numpy as np

from util import Pipeline, load_base_year_emp, load_input_tables, calc_gq
from util.elmer_helpers import read_from_elmer_geo, read_from_elmer
from steps.kitsap_unit_growth_adjustment import get_start_year, load_tables

p = Pipeline('configs_2026')

In [ ]:
with pd.HDFStore('data/pipeline.h5') as store:
    tables = store.keys()
tables

In [129]:
start_year = get_start_year(p)
df = load_tables(p, start_year)

In [ ]:
def split_housing_growth_targets(df):
    pop_growth = 'total_pop_chg'
    unit_growth_total = 'units_chg'
    df['ofm_hhsz'] = df['ofm_hhpop'] / df['ofm_hh']
    df['ofm_vacancy'] = 1 - df['ofm_hh'] / df['ofm_units']
    df['calc_hh_growth'] = df[pop_growth] / df['ofm_hhsz']
    df['calc_unit_growth'] = df['calc_hh_growth'] * (1 - df['ofm_vacancy'])
    df['pct_of_calc_unit_growth'] = df['calc_unit_growth'] / df['calc_unit_growth'].groupby(df['HousingJuris']).transform('sum')
    df[unit_growth_total] = df.groupby('HousingJuris')[unit_growth_total].transform('sum')
    df['units_chg'] = df['pct_of_calc_unit_growth'] * df[unit_growth_total]
    for juris in df['HousingJuris'].unique():
        mask = df['HousingJuris'] == juris
        df.loc[mask, 'units_chg'] = saferound(df.loc[mask, 'units_chg'], 0)
    df['units_chg'] = df['units_chg'].astype(int)
    df.to_csv('data/kitsap_targets_adj.csv', index=False)
    df = df[['target_id','name','total_pop_chg',
        'units_chg', 'emp_chg']]
    return df


df = split_housing_growth_targets(df)

In [130]:
df = df.loc[df['HousingJuris'] == 'Unincorporated'].copy()

In [131]:
unincorp_units_target = df['units_2044'].sum()
unincorp_units_target

np.float64(84485.0)

In [132]:
unincorp_pop_target = df['total_pop_2044'].sum()
unincorp_pop_target

np.int64(210609)

In [133]:
# calculate %GQ in start year population
df['ofm_gq_pct'] = df['ofm_gq'] / df['ofm_total_pop']

In [134]:
# use %GQ to calculate hhpop for target year
df['hhpop_2044'] = df['total_pop_2044'] * (1 - df['ofm_gq_pct'])

In [135]:
start_uninc_hhsz = df['ofm_hhpop'].sum() / df['ofm_hh'].sum()

In [136]:
start_uninc_vacancy = 1 - df['ofm_hh'].sum() / df['ofm_units'].sum()

In [137]:
unincorp_hh_target = unincorp_units_target * (1 - start_uninc_vacancy)

In [138]:
target_uninc_hhsz = df['hhpop_2044'].sum() / unincorp_hh_target
target_uninc_hhsz

np.float64(2.6150472985883613)

In [139]:
target_to_start_hhsz_ratio = target_uninc_hhsz / start_uninc_hhsz

In [141]:
df['ofm_hhsz'] = df['ofm_hhpop'] / df['ofm_hh']

In [142]:
df['hhsz_2044'] = df['ofm_hhsz'] * target_to_start_hhsz_ratio

In [143]:
df['prelim_hh_2044'] = df['hhpop_2044'] / df['hhsz_2044']

In [144]:
def normalize(df, target_total, value_col, new_col):
    df[new_col] = df[value_col] * (target_total / df[value_col].sum())
    return df

In [145]:
df = normalize(df, unincorp_hh_target, 'prelim_hh_2044', 'hh_2044')

In [146]:
# calculate start year vacancy rate
df['ofm_vacancy'] = 1 - df['ofm_hh'] / df['ofm_units']
df['prelim_units_2044'] = df['hh_2044'] * (1 - df['ofm_vacancy'])

In [147]:
df = normalize(df, unincorp_units_target, 'prelim_units_2044', 'norm_units_2044')

In [149]:
df[['name','ofm_units','prelim_units_2044','norm_units_2044']]

,name,ofm_units,prelim_units_2044,norm_units_2044
1,Bremerton UGA,4385.018045,4915.781944,5529.861871
2,Silverdale,8184.708333,11394.744830,12818.177389
4,Kingston,1173.000000,2374.655715,2671.297923
6,Port Orchard UGA,6065.000000,6759.160126,7603.515024
8,Poulsbo UGA,204.000000,520.780174,585.836080
9,Urban Unincorporated (Central Kitsap UGA),9006.305556,10283.725953,11568.369933
10,Rural Areas,42237.538956,38854.263635,43707.941780
